<a href="https://colab.research.google.com/github/jorgecaraveobalderas-beep/Class-Files/blob/main/Sesion10_Data_Profiling_Entregable_274702.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 10 — Entregable sobre Data Profiling

**Nombre completo: Jorge Alberto Caraveo Balderas**

**Matrícula: 274702**

---

Este notebook contiene las actividades a entregar de la Sesión 10 (Data Profiling), aplicadas sobre **datasets reales**: el catálogo de Netflix y el dataset de Customer Personality Analysis (Kaggle). Si aún no revisaste las explicaciones y ejemplos de cada tema, hazlo primero en el notebook `Sesion10_Data_Profiling_Actividad_Asincrona.ipynb`.

**Nota sobre los datos:** ambos datasets son reales — los problemas de calidad que vas a encontrar (nulos, categorías inconsistentes, valores fuera de rango) ya existían antes de que este notebook los usara. La única excepción está marcada explícitamente en la Actividad 3 y la Práctica integradora, donde se inyectan un par de filas/valores a propósito para poder practicar duplicados y ajuste de tipos con un resultado garantizado.

**Antes de entregar:** ejecuta "Reiniciar y ejecutar todo" para confirmar que tu notebook corre de principio a fin sin errores.

## Preparación

Ejecuta esta celda antes de empezar — descarga los dos datasets reales que vas a usar.

In [17]:
import pandas as pd

url_netflix = 'https://raw.githubusercontent.com/Vibe1990/Netflix-Project/main/netflix_title.csv'
url_marketing = 'https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv'

df_netflix = pd.read_csv(url_netflix)
df_marketing = pd.read_csv(url_marketing, sep=';')  # nota: este archivo usa punto y coma, no coma

print('Netflix:', df_netflix.shape)
print('Marketing:', df_marketing.shape)

Netflix: (7787, 12)
Marketing: (2240, 29)


---
## Actividad 1 — Renombrado y estandarización de columnas

*Dataset: Customer Personality Analysis*

Revisa los nombres de columna de `df_marketing` con `.columns`. Vas a notar una mezcla de convenciones reales: `Year_Birth` (con guion bajo), `Kidhome` (sin separador), `MntWines` (abreviado y sin separador).

**Trabaja sobre una copia** (`df_marketing_renombrado = df_marketing.copy()`) para no afectar las actividades siguientes, que usan los nombres originales. Aplica `.str.lower()` para al menos unificar mayúsculas/minúsculas, y usa `.rename()` para corregir manualmente los 2-3 nombres que la técnica automática no deja perfectos (por ejemplo, `mntwines` sigue sin ser ideal — decide tú el nombre final).

In [18]:


# Revisar el dataframee original con sus columnas
print(df_marketing.columns)

# Crear una copia para no afectar al dataframe original en base a la instrucción de la actividad 1
df_marketing_renombrado = df_marketing.copy()

# Utilizar .str.lower() para unificar el texto y sea representado únicamente en minusculas
df_marketing_renombrado.columns = df_marketing_renombrado.columns.str.lower()

# .rename(columns={...}) recibe un diccionario: nombre viejo (clave) -> nombre nuevo (valor). - Se renombra aquellas columnas donde el nombre inicial no sea muy claro para facilitar
# el análisis posterior
df_marketing_renombrado = df_marketing_renombrado.rename(columns={
    'kidhome': 'kid_home',
    'teenhome': 'teen_home',
    'mntwines': 'amount_wines',
    'mntfruits': 'amount_fruits',
    'mntmeatproducts': 'amount_meat_products',
    'mntfishproducts': 'amount_fish_products',
    'mntsweetproducts': 'amount_sweet_products',
    'mntgoldprods': 'amount_gold_products',
    'numdealspurchases': 'num_deals_purchases',
    'numwebpurchases': 'num_web_purchases',
    'numcatalogpurchases': 'num_catalog_purchases',
    'numstorepurchases': 'num_store_purchases',
    'numwebvisitsmonth': 'num_web_visits_month',
    'acceptedcmp3': 'accepted_cmp_3',
    'acceptedcmp4': 'accepted_cmp_4',
    'acceptedcmp5': 'accepted_cmp_5',
    'acceptedcmp1': 'accepted_cmp_1',
    'acceptedcmp2': 'accepted_cmp_2',
})


#Se imprime resultado del nuevo dataframe generado

df_marketing_renombrado.columns



Index(['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome',
       'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response'],
      dtype='object')


Index(['id', 'year_birth', 'education', 'marital_status', 'income', 'kid_home',
       'teen_home', 'dt_customer', 'recency', 'amount_wines', 'amount_fruits',
       'amount_meat_products', 'amount_fish_products', 'amount_sweet_products',
       'amount_gold_products', 'num_deals_purchases', 'num_web_purchases',
       'num_catalog_purchases', 'num_store_purchases', 'num_web_visits_month',
       'accepted_cmp_3', 'accepted_cmp_4', 'accepted_cmp_5', 'accepted_cmp_1',
       'accepted_cmp_2', 'complain', 'z_costcontact', 'z_revenue', 'response'],
      dtype='object')

---
## Actividad 2 — Ajuste de tipos: fechas con formato mixto

*Dataset: Netflix*

La columna `date_added` de `df_netflix` mezcla formatos reales: la mayoría son `"14-Aug-20"`, pero un grupo minoritario llega como `" August 4, 2017"` (con espacio inicial). Conviértela a tipo fecha usando `pd.to_datetime(..., format='mixed')`, que resuelve ambos formatos en la misma columna. Verifica con `.dtypes` y confirma cuántos valores nulos quedan después de la conversión (compara contra los nulos que ya traía antes de convertir).

In [19]:
#  Revisar la cantidad de datos nulos antes de la conversión

nulos_antes = df_netflix['date_added'].isnull().sum()
print("Cantidad de nulos antes de la conversion:", nulos_antes)
print("Tipo de dato de la columna antes de la conversion'date_added':", df_netflix['date_added'].dtype)

# Convertir a tipo fecha utilizando la función pd.to_datetime(..., format='mixed') sugerida en la actividad

df_netflix['date_added'] = pd.to_datetime(df_netflix['date_added'], format='mixed')

# Revisar la cantidad de datos nulos después de la conversión

nulos_despues = df_netflix['date_added'].isnull().sum()
print("Cantidad de nulos despues de la conversion:", nulos_despues)

# Verificar el tipo de dato

print("Tipo de dato de la columna despues de la conversion'date_added':", df_netflix['date_added'].dtype)

# Concluimos que la conversión fue exitosa debido a que se cuenta con la misma cantidad de valores nulos y el tipo de dato dejó de ser texto



Cantidad de nulos antes de la conversion: 10
Tipo de dato de la columna antes de la conversion'date_added': object
Cantidad de nulos despues de la conversion: 10
Tipo de dato de la columna despues de la conversion'date_added': datetime64[ns]


---
## Actividad 3 — Duplicados

*Dataset: Customer Personality Analysis — con 2 filas duplicadas inyectadas a propósito*

Este dataset real no trae duplicados de forma natural — para poder practicar, se insertan 2 copias de clientes ya existentes (ejecuta la celda siguiente).

In [20]:
df_marketing_dup = pd.concat([df_marketing, df_marketing.sample(2, random_state=7)], ignore_index=True)
print('Filas originales:', len(df_marketing))
print('Filas con duplicados inyectados:', len(df_marketing_dup))

Filas originales: 2240
Filas con duplicados inyectados: 2242


Sobre `df_marketing_dup`: cuenta los duplicados exactos con `.duplicated().sum()`, luego cuenta los duplicados por `ID` con `.duplicated(subset='ID').sum()` (deberían coincidir, ya que el `ID` es único por cliente). Elimínalos con `.drop_duplicates()` y confirma el número final de filas.

In [21]:
# Contar la cantidad de duplicados exactos con la función .duplicated().sum() sugeridad en la actividad

Cantidad_de_duplicados_exactos = df_marketing_dup.duplicated().sum()
print("Cantidad de duplicados exactos:", Cantidad_de_duplicados_exactos)

# Contar los duplicados por ID, con base en la función .duplicated(subset='ID').sum() sugerida

Cantidad_de_duplicados_por_ID = df_marketing_dup.duplicated(subset='ID').sum()
print("Cantidad de duplicados por ID:", Cantidad_de_duplicados_por_ID)

# Eliminar duplicados nuevamente considerando la función .drop_duplicates() sugerida

df_marketing_dup = df_marketing_dup.drop_duplicates()

# Confirmar el número final de filas

print("Numero final de filas:", len(df_marketing_dup))

# Concluimos que la función de drop_duplicates funcionó correctamente ya que el número final de filas coincide con la cifra de filas del dataset original


Cantidad de duplicados exactos: 2
Cantidad de duplicados por ID: 2
Numero final de filas: 2240


---
## Actividad 4 — Valores faltantes

*Dataset: Netflix*

Usa `.isnull().sum()` sobre `df_netflix` para ver cuántos valores faltan por columna. Luego, usa `.isnull().any(axis=1)` para filtrar solo las filas que tienen **al menos un** valor faltante en cualquier columna, y muestra cuántas filas son en total (`.sum()` sobre el resultado booleano).

In [22]:
# Calcular los valores faltantes por columna con la función sugerida .isnull().sum()

valores_faltantes_por_columna = df_netflix.isnull().sum()
print("Valores faltantes por columna:")
print(valores_faltantes_por_columna)

# Filtrar las filas que tienen por lo menos 1 valor nulo con la función sugerida .isnull().any(axis=1)

valores_faltantes_por_fila = df_netflix.isnull().any(axis=1)

# Cuantificar el total de filas con valor nulo con la función sugerida .sum()

filas_con_nulos = valores_faltantes_por_fila.sum()
print("Total de filas con valor faltante:", filas_con_nulos)

#Mostrar filas con valores nulos

df_netflix[valores_faltantes_por_fila]





Valores faltantes por columna:
show_id            0
type               0
title              0
director        2389
cast             718
country          507
date_added        10
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64
Total de filas con valor faltante: 2979


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,2020-08-14,2020,TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
11,s12,TV Show,1983,NaN,"Robert Więckiewicz, Maciej Musiał, Michalina O...","Poland, United States",2018-11-30,2018,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Dramas","In this dark alt-history thriller, a naïve law..."
12,s13,TV Show,1994,Diego Enrique Osorno,NaN,Mexico,2019-05-17,2019,TV-MA,1 Season,"Crime TV Shows, Docuseries, International TV S...",Archival video and new interviews examine Mexi...
16,s17,TV Show,09-Feb,NaN,"Shahd El Yaseen, Shaila Sabt, Hala, Hanadi Al-...",NaN,2019-03-20,2018,TV-14,1 Season,"International TV Shows, TV Dramas","As a psychology professor faces Alzheimer's, h..."
19,s20,Movie,'89,NaN,"Lee Dixon, Ian Wright, Paul Merson",United Kingdom,2018-05-16,2017,TV-PG,87 min,Sports Movies,"Mixing old footage with interviews, this is th..."
...,...,...,...,...,...,...,...,...,...,...,...,...
7777,s7778,TV Show,Zombie Dumb,NaN,NaN,NaN,2019-07-01,2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
7779,s7780,TV Show,Zona Rosa,NaN,"Manu NNa, Ana Julia Yeyé, Ray Contreras, Pablo...",Mexico,2019-11-26,2019,TV-MA,1 Season,"International TV Shows, Spanish-Language TV Sh...",An assortment of talent takes the stage for a ...
7784,s7785,Movie,Zulu Man in Japan,NaN,Nasty C,NaN,2020-09-25,2019,TV-MA,44 min,"Documentaries, International Movies, Music & M...","In this documentary, South African rapper Nast..."
7785,s7786,TV Show,Zumbo's Just Desserts,NaN,"Adriano Zumbo, Rachel Khoo",Australia,2020-10-31,2019,TV-PG,1 Season,"International TV Shows, Reality TV",Dessert wizard Adriano Zumbo looks for the nex...


---
## Actividad 5 — Completitud como porcentaje

*Dataset: Netflix*

Con los mismos nulos de la Actividad 4, calcula la completitud en porcentaje por columna: `(1 - nulos / total_filas) * 100`. ¿Qué columna tiene la completitud más baja? Escribe la respuesta en una línea.

In [23]:
# Calcular el porcentaje de completitud por columna con la función sugerida (1 - nulos / total_filas) * 100.

porcentaje_completitud_por_columna = (1 - df_netflix.isnull().sum() / len(df_netflix)) * 100
print("Porcentaje de valores por completitud por columna:")
print(porcentaje_completitud_por_columna)

# Encontrar la columna con la completitud más baja

columna_menor_completitud = porcentaje_completitud_por_columna.idxmin()
print("La columna con la menor completitud es:", columna_menor_completitud)




Porcentaje de valores por completitud por columna:
show_id         100.000000
type            100.000000
title           100.000000
director         69.320663
cast             90.779504
country          93.489149
date_added       99.871581
release_year    100.000000
rating           99.910107
duration        100.000000
listed_in       100.000000
description     100.000000
dtype: float64
La columna con la menor completitud es: director


---
## Actividad 6 — Exploración categórica

*Dataset: Customer Personality Analysis*

Aplica `.value_counts()` sobre la columna `Marital_Status` de `df_marketing`. Vas a encontrar, junto a las categorías esperadas (`Married`, `Single`, `Together`, `Divorced`, `Widow`), tres valores que claramente son errores de captura reales: `Alone`, `Absurd` y `YOLO`. Decide y justifica en una línea: ¿los eliminarías, los reclasificarías (por ejemplo, `Alone` → `Single`), o los dejarías así? No hay una única respuesta correcta — lo que importa es la justificación.

In [24]:
# Utilizar función sugerida .value_counts()  por la acitivdad para encontrar las categorías de la columna marital_status

print(df_marketing['Marital_Status'].value_counts())

# Justificación para los datos claramente erróneos

porcentaje_por_categoria = (df_marketing['Marital_Status'].value_counts(normalize=True) * 100) #Calcular porcentaje por categoría
print(porcentaje_por_categoria)

print("Debido a que el impacto de estas 3 categorias es minima (menor al 1%) los eliminaria del dataframe ya que estos no influirian en los analisis posteriores")




Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64
Marital_Status
Married     38.571429
Together    25.892857
Single      21.428571
Divorced    10.357143
Widow        3.437500
Alone        0.133929
Absurd       0.089286
YOLO         0.089286
Name: proportion, dtype: float64
Debido a que el impacto de estas 3 categorias es minima (menor al 1%) los eliminaria del dataframe ya que estos no influirian en los analisis posteriores


---
## Actividad 7 — Consistencia de formato/patrón

*Dataset: Netflix*

La columna `show_id` debería seguir siempre el patrón: la letra `s` seguida de uno o más dígitos (`s1`, `s2`, ..., `s8807`). Verifica con `.str.match(r'^s\d+$')` si todos los valores cumplen esta convención. Reporta el porcentaje de cumplimiento.

In [25]:
# Evaluar si los valores cumplen con los patrones mencionados en la actividad para la columna show_id

cumplimiento = df_netflix['show_id'].str.match(r'^s\d+$')

# Calcular porcentaje

porcentaje_cumplimiento = cumplimiento.mean() * 100
print("Porcentaje de cumplimiento:", porcentaje_cumplimiento)


Porcentaje de cumplimiento: 100.0


---
## Actividad 8 — `.info()` y `.describe()`

*Dataset: Customer Personality Analysis*

Ejecuta `.describe()` sobre la columna `Year_Birth` de `df_marketing` (puedes hacerlo con `df_marketing[['Year_Birth']].describe()`). Observa el valor mínimo (`min`). ¿Tiene sentido ese año de nacimiento? Filtra el DataFrame para mostrar las filas con los años de nacimiento más antiguos y decide, en una línea, si los considerarías un error de captura.

In [26]:
# Ejecutar función sugerida .describe()
df_marketing[['Year_Birth']].describe()

# Fitrar dataframe para observar los años de nacimiento más antiguos

antiguos = df_marketing[df_marketing['Year_Birth'] < 1900]
print(antiguos)

print("Considero que sons errores de captura ya que adicional a que son personas con mas de 120 anos de vida, se puede verificar por el resto de informacion de estos 2 ID especificos que son personas que cuenta con un ingreso economico, lo cual quiere decir que se desempenan activamente en un rol de trabajo y el nivel de educacion para uno es de pregado")




        ID  Year_Birth Education Marital_Status   Income  Kidhome  Teenhome  \
239  11004        1893  2n Cycle         Single  60182.0        0         1   
339   1150        1899       PhD       Together  83532.0        0         0   

    Dt_Customer  Recency  MntWines  ...  NumWebVisitsMonth  AcceptedCmp3  \
239  2014-05-17       23         8  ...                  4             0   
339  2013-09-26       36       755  ...                  1             0   

     AcceptedCmp4  AcceptedCmp5  AcceptedCmp1  AcceptedCmp2  Complain  \
239             0             0             0             0         0   
339             0             1             0             0         0   

     Z_CostContact  Z_Revenue  Response  
239              3         11         0  
339              3         11         0  

[2 rows x 29 columns]
Considero que sons errores de captura ya que adicional a que son personas con mas de 120 anos de vida, se puede verificar por el resto de informacion de estos 2 ID 

---
## Actividad 9 — Práctica integradora: checklist de profiling

*Dataset: muestra real de Customer Personality Analysis, con 2 elementos inyectados y marcados a propósito (una fila duplicada y un valor de tipo incorrecto) para poder practicar el checklist completo con un resultado garantizado.*

Aplica el checklist completo, en orden, sobre `df_practica`:

1. Revisa `.dtypes` e identifica qué columna tiene un problema de tipo, corrígela con `pd.to_numeric(errors='coerce')`
2. Cuenta las filas duplicadas y elimínalas con `.drop_duplicates()`
3. Cuenta los valores faltantes por columna con `.isnull().sum()` (incluyendo el que se generó en el paso 1)
4. Revisa `.unique()` sobre `Marital_Status` y decide si necesita normalización

Al final, escribe un breve "reporte de profiling" (3-4 líneas) resumiendo qué encontraste y qué decidiste.

In [27]:
# Muestra real con 2 elementos inyectados (marcados abajo)
df_practica = df_marketing.sample(15, random_state=3).reset_index(drop=True).copy()

# Elemento inyectado 1: una fila duplicada
df_practica = pd.concat([df_practica, df_practica.iloc[[2]]], ignore_index=True)

# Elemento inyectado 2: un valor de tipo incorrecto en Income
df_practica['Income'] = df_practica['Income'].astype(object)
df_practica.loc[5, 'Income'] = 'sesenta mil'

df_practica[['ID', 'Marital_Status', 'Income']]

,ID,Marital_Status,Income
0,5788,Together,46053.0
1,7930,Single,26877.0
2,4557,Together,22070.0
3,9964,Single,61825.0
4,1168,Married,72159.0
5,5314,Together,sesenta mil
6,9665,Divorced,54237.0
7,6182,Together,26646.0
8,922,Married,31086.0
9,4427,Single,83257.0


In [28]:

# Paso 1 — ajuste de tipos. Revisa .dtypes e identifica qué columna tiene un problema de tipo, corrígela con pd.to_numeric(errors='coerce')

# Asegurar cuál es la columna con un tipo erróneo

print(df_practica.dtypes) # Income arroja como "object" cuando debería ser "cuantitativo"

df_practica['Income'] = pd.to_numeric(df_practica['Income'], errors='coerce') # Se transforma a tipo "float"
df_practica.dtypes



ID                      int64
Year_Birth              int64
Education              object
Marital_Status         object
Income                 object
Kidhome                 int64
Teenhome                int64
Dt_Customer            object
Recency                 int64
MntWines                int64
MntFruits               int64
MntMeatProducts         int64
MntFishProducts         int64
MntSweetProducts        int64
MntGoldProds            int64
NumDealsPurchases       int64
NumWebPurchases         int64
NumCatalogPurchases     int64
NumStorePurchases       int64
NumWebVisitsMonth       int64
AcceptedCmp3            int64
AcceptedCmp4            int64
AcceptedCmp5            int64
AcceptedCmp1            int64
AcceptedCmp2            int64
Complain                int64
Z_CostContact           int64
Z_Revenue               int64
Response                int64
dtype: object


,0
ID,int64
Year_Birth,int64
Education,object
Marital_Status,object
Income,float64
Kidhome,int64
Teenhome,int64
Dt_Customer,object
Recency,int64
MntWines,int64


In [29]:

# Paso 2 — duplicados. Cuenta las filas duplicadas y elimínalas con .drop_duplicates()

# Contar filas duplicadas

print(df_practica.duplicated().sum())

# Eliminar duplicados

df_practica = df_practica.drop_duplicates()
df_practica





1


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5788,1972,Graduation,Together,46053.0,1,0,2013-02-17,46,56,...,5,0,0,0,0,0,0,3,11,0
1,7930,1969,Master,Single,26877.0,0,0,2013-08-19,74,101,...,6,0,0,0,0,0,0,3,11,0
2,4557,1970,Graduation,Together,22070.0,1,0,2013-10-04,11,10,...,6,0,0,0,0,0,0,3,11,0
3,9964,1979,Graduation,Single,61825.0,0,1,2013-08-07,56,162,...,4,0,0,0,0,0,0,3,11,0
4,1168,1978,PhD,Married,72159.0,0,0,2012-10-29,62,322,...,2,0,0,0,0,0,0,3,11,0
5,5314,1951,Graduation,Together,NaN,0,1,2013-03-18,69,270,...,8,0,0,0,0,0,0,3,11,0
6,9665,1957,PhD,Divorced,54237.0,0,1,2013-04-27,48,267,...,6,0,0,0,0,0,0,3,11,0
7,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,...,6,0,0,0,0,0,0,3,11,0
8,922,1979,2n Cycle,Married,31086.0,1,1,2013-05-04,79,16,...,8,1,0,0,0,0,0,3,11,0
9,4427,1995,2n Cycle,Single,83257.0,0,0,2012-09-18,56,536,...,6,1,0,1,0,0,1,3,11,1


In [30]:
# Paso 3 — valores faltantes, Cuenta los valores faltantes por columna con .isnull().sum() (incluyendo el que se generó en el paso 1)

df_practica.isnull().sum()


,0
ID,0
Year_Birth,0
Education,0
Marital_Status,0
Income,1
Kidhome,0
Teenhome,0
Dt_Customer,0
Recency,0
MntWines,0


In [31]:
# Paso 4 — exploración categórica. Revisa .unique() sobre Marital_Status y decide si necesita normalización


print(df_practica['Marital_Status'].unique())
print("No requiere normalizacion ya que muestra unicamente 4 categorias las cuales no tienen indicios de errores tipograficos. El tipo de variable es objeto, lo cual hace sentido con el resultado obtenido. y cuenta con una sintaxis facil de leer")



['Together' 'Single' 'Married' 'Divorced']
No requiere normalizacion ya que muestra unicamente 4 categorias las cuales no tienen indicios de errores tipograficos. El tipo de variable es objeto, lo cual hace sentido con el resultado obtenido. y cuenta con una sintaxis facil de leer



**Tu reporte de profiling:**

*(Escribe aquí tu resumen de 3-4 líneas)*

Se obtuvo la calidad de datos deseada, ya que:

1.  Se detectó el error en la columna "Income" la cual fue corregida exitosamente y convertida a categoría numérica
2.  Se eliminaron las filas duplicadas de manera correcta, dando como resultado un dataframe limpio para un análisis
3.  Se identificaron correctamente los valores faltantes por columna, así manteniendo un orden para la ejecución del análisis descriptivo
